# Course 9: Ten Years of Deep Learning Ideas
## Session 5: Self-supervised learning

**Goal:** train SimCLR, SimSiam and a tiny masked autoencoder without labels, detect collapse, and compare their features with a supervised model using a linear probe on 10% of the labels.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Setup, data and augmentations
We treat scikit-learn's 8x8 digits as images. Self-supervised training uses all 1,347 training images **without labels**; afterwards we pretend that only 10% of the labels (134 images) are known. Two random **views** of each image are made with small rotations (up to 15 degrees), scaling (up to 10%), shifts (up to 1 pixel) and noise. Real self-supervised learning uses millions of photos, crops and colour changes; the ideas are the same.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

torch.set_num_threads(2)          # tiny models run as fast on 2 threads, and shared machines stay responsive
threadpool_limits(2)
torch.set_flush_denormal(True)
torch.manual_seed(0); np.random.seed(0)

digits = load_digits()
X_all = torch.tensor(digits.data / 16.0, dtype=torch.float32).view(-1, 1, 8, 8)
y_all = torch.tensor(digits.target)
Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.25, random_state=0, stratify=y_all)
labeled, _ = train_test_split(np.arange(len(ytr)), train_size=0.1, random_state=0, stratify=ytr.numpy())
print(f"{len(Xtr)} unlabeled training images, {len(labeled)} of them with labels, {len(Xte)} test images")


def augment(x, g):
    """Random rotation (+-15 deg), scale (+-10%), shift (+-1 pixel) and Gaussian noise, for a batch (B, 1, 8, 8)."""
    B = len(x)
    angle = (torch.rand(B, generator=g) - 0.5) * 2 * math.radians(15)
    scale = 1 + (torch.rand(B, generator=g) - 0.5) * 0.2
    shift = (torch.rand(B, 2, generator=g) - 0.5) * 0.5           # grid units: 0.25 is one pixel of 8
    cos, sin = torch.cos(angle) / scale, torch.sin(angle) / scale
    theta = torch.stack([torch.stack([cos, -sin, shift[:, 0]], 1), torch.stack([sin, cos, shift[:, 1]], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, align_corners=False) + 0.1 * torch.randn(x.shape, generator=g)


def linear_probe(f_train, f_test):
    """Logistic regression on frozen features, trained on the 134 labeled images only."""
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    clf.fit(f_train[labeled], ytr.numpy()[labeled])
    return clf.score(f_test, yte.numpy())


g = torch.Generator().manual_seed(0)
fig, ax = plt.subplots(2, 8, figsize=(9, 2.6))
for j in range(8):
    v1, v2 = augment(Xtr[j:j + 1], g), augment(Xtr[j:j + 1], g)
    ax[0, j].imshow(v1[0, 0], cmap="gray"); ax[1, j].imshow(v2[0, 0], cmap="gray")
    ax[0, j].axis("off"); ax[1, j].axis("off")
plt.suptitle("Two random views of the same eight digits (top and bottom rows)", fontsize=9)
plt.tight_layout(); plt.show()

### 2. Baselines: what can 134 labels do alone?
All methods share one small encoder: an MLP from the 64 pixels to a 128-dimensional representation h. We compare against (a) a linear probe on raw pixels, (b) a linear probe on a random, untrained encoder, and (c) the same encoder trained with labels (and the same augmentations) on the 134 labeled images only, and on all 1,347 as an upper bound.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, rep=128):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(64, 256), nn.BatchNorm1d(256), nn.ReLU(),
                                 nn.Linear(256, rep), nn.BatchNorm1d(rep), nn.ReLU())
    def forward(self, x):
        return self.net(x)


def features(module):
    module.eval()
    with torch.no_grad():
        return module(Xtr).numpy(), module(Xte).numpy()


def supervised(index, steps=400, seed=0):
    torch.manual_seed(seed)
    enc, head = Encoder(), nn.Linear(128, 10)
    opt = torch.optim.Adam(list(enc.parameters()) + list(head.parameters()), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    Xl, yl = Xtr[index], ytr[index]
    for step in range(steps):
        i = torch.randint(len(Xl), (64,), generator=g)
        loss = F.cross_entropy(head(enc(augment(Xl[i], g))), yl[i])
        opt.zero_grad(); loss.backward(); opt.step()
    enc.eval(); head.eval()
    with torch.no_grad():
        return (head(enc(Xte)).argmax(1) == yte).float().mean().item()


scores = {}
scores["raw pixels, linear probe"] = linear_probe(Xtr.flatten(1).numpy(), Xte.flatten(1).numpy())
torch.manual_seed(0)
scores["random encoder, linear probe"] = linear_probe(*features(Encoder()))
scores["supervised, 10% of labels"] = supervised(labeled)
scores["supervised, all labels (upper bound)"] = supervised(np.arange(len(ytr)))
for name, acc in scores.items():
    print(f"{name:38s} test accuracy {acc:.3f}")

### 3. SimCLR: contrastive learning with InfoNCE
SimCLR (Chen et al. 2020) makes two views of every image in a batch of N, passes all 2N through the encoder and a small projection head, and normalizes the outputs z to unit length. For each view, the other view of the same image is the positive and the other 2N − 2 views are negatives. The InfoNCE loss (van den Oord et al. 2018) is a cross-entropy over cosine similarities divided by a temperature τ:

`loss_i = -log( exp(z_i·z_j / τ) / Σ_{k≠i} exp(z_i·z_k / τ) )`, where j is the positive of i.

After training we throw the projection head away and probe the encoder's representation h.

In [ ]:
def mlp_head(d_in, d_out, hidden=128):
    return nn.Sequential(nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Linear(hidden, d_out))


def train_simclr(steps=400, bs=256, tau=0.2, seed=0):
    torch.manual_seed(seed)
    enc, head = Encoder(), mlp_head(128, 64)
    opt = torch.optim.Adam(list(enc.parameters()) + list(head.parameters()), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    targets = torch.cat([torch.arange(bs, 2 * bs), torch.arange(0, bs)])     # view i's positive is view i +- bs
    for step in range(steps):
        x = Xtr[torch.randint(len(Xtr), (bs,), generator=g)]
        z = F.normalize(head(enc(torch.cat([augment(x, g), augment(x, g)]))), dim=1)
        sim = z @ z.T / tau
        sim.fill_diagonal_(float("-inf"))                                  # a view is not its own negative
        loss = F.cross_entropy(sim, targets)
        opt.zero_grad(); loss.backward(); opt.step()
    return enc, loss.item()

t0 = time.time()
simclr_enc, simclr_loss = train_simclr()
scores["SimCLR, linear probe"] = linear_probe(*features(simclr_enc))
print(f"SimCLR: final InfoNCE loss {simclr_loss:.3f} (chance level log(511) = {math.log(511):.3f}); "
      f"linear probe {scores['SimCLR, linear probe']:.3f} ({time.time() - t0:.0f} s)")

### 4. SimSiam: no negatives, and why the stop-gradient matters
SimSiam (Chen & He 2021) drops the negatives. A predictor MLP p tries to predict one view's projection z₂ from the other's, p(z₁), and the loss is the negative cosine similarity. A **constant** output would give the perfect loss of −1 for every image: this is **collapse**. SimSiam's defence is the stop-gradient: z₂ is treated as a fixed target (`.detach()`). We train it with and without the stop-gradient and watch the standard deviation of the normalized outputs across the batch: about 1/√64 = 0.125 when the outputs are spread out over 64 dimensions, near 0 when they collapse.

In [ ]:
def train_simsiam(stop_gradient=True, steps=400, bs=256, seed=0):
    torch.manual_seed(seed)
    enc, projector, predictor = Encoder(), mlp_head(128, 64), mlp_head(64, 64, hidden=32)
    params = list(enc.parameters()) + list(projector.parameters()) + list(predictor.parameters())
    opt = torch.optim.SGD(params, lr=0.05, momentum=0.9, weight_decay=1e-4)
    g = torch.Generator().manual_seed(seed)
    spread = []
    for step in range(steps):
        x = Xtr[torch.randint(len(Xtr), (bs,), generator=g)]
        z1, z2 = projector(enc(augment(x, g))), projector(enc(augment(x, g)))
        p1, p2 = predictor(z1), predictor(z2)
        t1, t2 = (z1.detach(), z2.detach()) if stop_gradient else (z1, z2)
        loss = -(F.cosine_similarity(p1, t2).mean() + F.cosine_similarity(p2, t1).mean()) / 2
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 10 == 0:
            spread.append(F.normalize(z1.detach(), dim=1).std(0).mean().item())
    return enc, projector, loss.item(), spread

siam = {}
for sg in [True, False]:
    enc, projector, final_loss, spread = train_simsiam(sg)
    h_tr, h_te = features(enc)
    projector.eval()
    with torch.no_grad():
        z_tr, z_te = projector(torch.tensor(h_tr)).numpy(), projector(torch.tensor(h_te)).numpy()
    siam[sg] = spread
    name = "with stop-gradient" if sg else "without stop-gradient"
    scores[f"SimSiam {name}, linear probe"] = linear_probe(h_tr, h_te)
    print(f"SimSiam {name:22s}: final loss {final_loss:.3f} | spread of normalized z {spread[-1]:.3f} | "
          f"probe on h {linear_probe(h_tr, h_te):.3f} | probe on z {linear_probe(z_tr, z_te):.3f}")

plt.figure(figsize=(6, 3.2))
for sg, spread in siam.items():
    plt.plot(np.arange(len(spread)) * 10, spread, label="with stop-gradient" if sg else "without stop-gradient")
plt.axhline(1 / math.sqrt(64), color="gray", ls=":", label="1/sqrt(64)")
plt.title("SimSiam: spread of the normalized outputs z")
plt.xlabel("step"); plt.ylabel("std across the batch"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 5. A tiny masked autoencoder (MAE)
MAE (He et al. 2022) hides 75% of the patches, encodes only the visible ones with a transformer, and a light decoder reconstructs the hidden pixels. Here: 2x2 patches (16 per digit), 4 visible and 12 hidden, a 2-layer encoder of width 64 and a 1-layer decoder of width 32. The loss is the squared error on the hidden patches only. For the probe we feed all 16 patches to the encoder and average its outputs.

In [ ]:
class TinyMAE(nn.Module):
    def __init__(self, d=64, d_dec=32):
        super().__init__()
        layer = lambda w: nn.TransformerEncoderLayer(w, 4, 2 * w, dropout=0.0, batch_first=True, norm_first=True)
        self.embed, self.pos = nn.Linear(4, d), nn.Parameter(torch.randn(1, 16, d) * 0.02)
        self.encoder = nn.TransformerEncoder(layer(d), 2, enable_nested_tensor=False)
        self.enc_norm = nn.LayerNorm(d)
        self.to_dec, self.mask_token = nn.Linear(d, d_dec), nn.Parameter(torch.zeros(1, 1, d_dec))
        self.dec_pos = nn.Parameter(torch.randn(1, 16, d_dec) * 0.02)
        self.decoder = nn.TransformerEncoder(layer(d_dec), 1, enable_nested_tensor=False)
        self.out = nn.Linear(d_dec, 4)

    @staticmethod
    def patchify(x):                                            # (B, 1, 8, 8) -> (B, 16 patches, 4 pixels)
        return x.unfold(2, 2, 2).unfold(3, 2, 2).reshape(len(x), 16, 4)

    def encode(self, patches, keep=None):
        h = self.embed(patches) + self.pos
        if keep is not None:                                    # only the visible patches enter the encoder
            h = torch.gather(h, 1, keep[..., None].expand(-1, -1, h.shape[-1]))
        return self.enc_norm(self.encoder(h))

    def forward(self, x, g, mask_ratio=0.75):
        p = self.patchify(x)
        B, n_keep = len(p), int(16 * (1 - mask_ratio))
        keep = torch.rand(B, 16, generator=g).argsort(1)[:, :n_keep]
        h = self.to_dec(self.encode(p, keep))
        full = self.mask_token.expand(B, 16, -1).clone()
        full.scatter_(1, keep[..., None].expand(-1, -1, h.shape[-1]), h)    # visible tokens back in place
        recon = self.out(self.decoder(full + self.dec_pos))
        hidden = torch.ones(B, 16).scatter_(1, keep, 0.0)
        loss = (((recon - p) ** 2).mean(-1) * hidden).sum() / hidden.sum()
        return loss, recon, keep

t0 = time.time()
torch.manual_seed(0)
mae = TinyMAE()
opt = torch.optim.AdamW(mae.parameters(), lr=1e-3, weight_decay=0.05)
g = torch.Generator().manual_seed(0)
for step in range(600):
    loss, _, _ = mae(Xtr[torch.randint(len(Xtr), (256,), generator=g)], g)
    opt.zero_grad(); loss.backward(); opt.step()
mae.eval()
with torch.no_grad():
    m_tr = mae.encode(TinyMAE.patchify(Xtr)).mean(1).numpy()
    m_te = mae.encode(TinyMAE.patchify(Xte)).mean(1).numpy()
    torch.manual_seed(0); untrained = TinyMAE().eval()
    r_tr = untrained.encode(TinyMAE.patchify(Xtr)).mean(1).numpy()
    r_te = untrained.encode(TinyMAE.patchify(Xte)).mean(1).numpy()
scores["MAE, linear probe"] = linear_probe(m_tr, m_te)
print(f"MAE: reconstruction loss on hidden patches {loss.item():.4f}; linear probe {scores['MAE, linear probe']:.3f} "
      f"(untrained MAE encoder: {linear_probe(r_tr, r_te):.3f}) ({time.time() - t0:.0f} s)")

with torch.no_grad():
    _, recon, keep = mae(Xte[:6], torch.Generator().manual_seed(5))
p = TinyMAE.patchify(Xte[:6])
visible = torch.zeros_like(p); visible.scatter_(1, keep[..., None].expand(-1, -1, 4), torch.gather(p, 1, keep[..., None].expand(-1, -1, 4)))
filled = recon.clone(); filled.scatter_(1, keep[..., None].expand(-1, -1, 4), torch.gather(p, 1, keep[..., None].expand(-1, -1, 4)))
unpatch = lambda q: q.reshape(-1, 4, 4, 2, 2).permute(0, 1, 3, 2, 4).reshape(-1, 8, 8)
fig, ax = plt.subplots(3, 6, figsize=(7, 3.5))
for j in range(6):
    for r, (img, title) in enumerate([(Xte[j, 0], "original"), (unpatch(visible)[j], "25% visible"), (unpatch(filled)[j], "reconstruction")]):
        ax[r, j].imshow(img, cmap="gray", vmin=0, vmax=1); ax[r, j].axis("off")
        if j == 0: ax[r, j].set_title(title, fontsize=8, loc="left")
plt.tight_layout(); plt.show()

### 6. Summary

In [ ]:
for name, acc in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f"{name:45s} {acc:.3f}")

### 7. Frozen DINOv2 features on real images (NOT RUN here)
At scale, self-supervised features are strong enough to use frozen. The cell follows the DINOv2 repository's `torch.hub` interface; check it before running.

In [ ]:
# NOT RUN: needs downloads (DINOv2 weights through torch.hub, CIFAR-10 through torchvision) and preferably a GPU.
import torch, torchvision, torchvision.transforms as T
from sklearn.linear_model import LogisticRegression
model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").cuda().eval()     # ViT-S/14, 21M parameters
tf = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
def embed(train):
    data = torchvision.datasets.CIFAR10("data", train=train, download=True, transform=tf)
    feats, labels = [], []
    with torch.no_grad():
        for x, y in torch.utils.data.DataLoader(data, batch_size=256, num_workers=2):
            feats.append(model(x.cuda()).cpu()); labels.append(y)                 # the class-token embedding
    return torch.cat(feats).numpy(), torch.cat(labels).numpy()
(f_tr, y_tr), (f_te, y_te) = embed(True), embed(False)
clf = LogisticRegression(max_iter=2000).fit(f_tr[:5000], y_tr[:5000])           # 10% of the CIFAR-10 labels
print(f"CIFAR-10 linear probe on frozen DINOv2 features, 10% of labels: {clf.score(f_te, y_te):.3f}")

### TODO
1. Change the SimCLR temperature to 0.05 and 1.0. How does the linear probe change?
2. Remove the rotation and scaling from `augment` (keep the noise only) and retrain SimCLR. What do you learn about the role of augmentations?
3. Complete the Barlow Twins loss below (Zbontar et al. 2021), then train it like SimSiam (without a predictor).

In [ ]:
# TODO: Barlow Twins. z1, z2: (B, D) projections of two views. Standardize each dimension over the batch, form the
# D x D cross-correlation C = z1n.T @ z2n / B, and return sum_i (1 - C_ii)^2 + lam * sum_{i != j} C_ij^2.
def barlow_twins_loss(z1, z2, lam=5e-3):
    return None          # replace with your implementation

out = barlow_twins_loss(torch.randn(256, 64), torch.randn(256, 64))
print("not done yet" if out is None else f"loss for two unrelated random batches: {float(out):.1f} (about 64 is expected)")

### What to notice
- **Baselines.** With only 134 labels, logistic regression on raw pixels reached 0.893 and the supervised encoder 0.927; with all 1,347 labels the supervised encoder reached 0.978. A random, untrained encoder gave 0.862.
- **SimCLR** features gave 0.933 with the same 134 labels, about the same as the supervised model trained on those labels (0.927). With one seed, treat this as a tie. SimCLR's advantage is that it could also learn from the 1,213 images without labels.
- **Collapse.** Without the stop-gradient, SimSiam's loss went to -0.997, almost the perfect -1, while the spread of its normalized outputs fell from 0.105 to 0.010: the outputs became nearly the same for every image. A linear probe on those outputs dropped to 0.684 (0.898 with the stop-gradient). The encoder underneath stopped learning rather than being destroyed: its probe was 0.902 against 0.918. So measure collapse at the layer the loss sees, and remember that a very low loss can mean a useless solution.
- **MAE.** The tiny MAE learned to fill in hidden patches (loss 0.0527, blurry but recognizable digits), but its linear probe was only 0.787, below raw pixels (0.893). He et al. (2022) also found MAE features weaker than contrastive ones under a linear probe and stronger once part of the network is fine-tuned; at this tiny scale (2x2 patches of an 8x8 image) there is also little structure to learn.
- **Self-supervised against supervised.** At this scale self-supervised learning did not beat supervised learning: SimCLR tied the 10%-label model, the other methods were below it, and all were well below the 0.978 reached with all labels. Results such as DINOv2, where frozen self-supervised features match or beat supervised ones for transfer, need very large models, data and compute (Oquab et al. 2024).
- **Limits.** One seed, tiny MLP encoders, 8x8 images and 400 to 600 training steps.